## RAG with PDF Data extraction to give context to LLM

In [1]:
from dotenv import load_dotenv
import os

load = load_dotenv('./../.env', override=True)

#print(os.getenv('LANGSMITH_PROJECT'))

In [2]:
from langchain_ollama import ChatOllama
import os

ollama_cloud_llm = ChatOllama(
    base_url="http://localhost:11434/",  # Ollama cloud endpoint
    model="devstral-small-2:24b-cloud", #gemini-3-flash-preview:cloud #qwen3.5:cloud
    temperature=0.5,
    max_tokens=450,
    headers={
        "Authorization": f"Bearer {os.getenv('OLLAMA_CLOUD_API_KEY')}"  # Cloud auth
    }
)

ollama_local_llm = ChatOllama(
    base_url="http://localhost:11434/",
    model="llama3.2:latest",
    temperature=0.5,
    max_tokens=450
)

### 1. Extraction the PDF files

In [3]:
from langchain_community.document_loaders import PyPDFLoader

pdf1 = "./attention.pdf"
pdf2 = "./LLMForgetting.pdf"
pdf3 = "./TestingAndEvaluatingLLM.pdf"

pdfFiles = [pdf1, pdf2, pdf3]

documents = []

for pdf in pdfFiles:
    loader = PyPDFLoader(pdf)
    docs = loader.load()
    documents.extend(docs)

print(f"Total documents loaded: {len(documents)}")

Total documents loaded: 253


In [4]:
print(f"Get document content by index 0: {documents[1]}")

Get document content by index 0: page_content='1 Introduction
Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks
in particular, have been firmly established as state of the art approaches in sequence modeling and
transduction problems such as language modeling and machine translation [ 35, 2, 5]. Numerous
efforts have since continued to push the boundaries of recurrent language models and encoder-decoder
architectures [38, 24, 15].
Recurrent models typically factor computation along the symbol positions of the input and output
sequences. Aligning the positions to steps in computation time, they generate a sequence of hidden
states ht, as a function of the previous hidden state ht−1 and the input for position t. This inherently
sequential nature precludes parallelization within training examples, which becomes critical at longer
sequence lengths, as memory constraints limit batching across examples. Recent work has achieved
significant improve

### 2. Text Splitting

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, add_start_index=True)

all_splits = text_splitter.split_documents(documents)

len(all_splits)

640

### 3. Embedding

In [ ]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="llama3.2:latest")

vector_1 = embeddings.embed_query(all_splits[0].page_content)
vector_2 = embeddings.embed_query(all_splits[1].page_content)

assert len(vector_1) == len(vector_2)

print(f"Length of embedding vector: {len(vector_1)}")

ResponseError: model "lama3.2:latest" not found, try pulling it first (status code: 404)